In [2]:
# Kaggle setup
import os
print('Running on Kaggle:', os.path.exists('/kaggle'))

Running on Kaggle: True


# Text-Based Sentiment Analysis Using BERT and Sub-Word Embeddings
**Author:** Tania Marietta Anto  
**Supervisor:** Dr. David Williams  
**Programme:** M.Sc. Artificial Intelligence — Dublin Business School  

---

This notebook implements a two-phase sentiment analysis system:
- **Phase 1** — Baseline sentiment classification using `bert-base-cased`
- **Phase 2** — Multi-task BERT with sarcasm detection and contrastive loss


---
## Phase 1: Baseline Sentiment Analysis Model

A `bert-base-cased` model is fine-tuned on the Sentiment140 dataset to classify tweets as positive or negative. This establishes a performance benchmark and reveals the model's limitations on pragmatic language such as sarcasm and irony.


### 1.1 Imports

In [3]:
!pip install transformers scikit-learn pandas -q

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import BertTokenizer, BertForSequenceClassification, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils import resample

def set_seed(seed=42):
    """Fixes randomness across Python, NumPy, and PyTorch (CPU + CUDA)
    so results are reproducible, not just the sklearn splits."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/vocab.py
/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/vocab.json
/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/train_data.csv
/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/test_data.csv
/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/Exploring Data.ipynb


### 1.2 Load and Prepare the Sentiment140 Dataset

In [5]:
!find /kaggle/input -name "*.csv"

/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/train_data.csv
/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/test_data.csv


In [6]:
import pandas as pd

train_path = "/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/train_data.csv"

df = pd.read_csv(train_path)

print(df.head())
print(df.columns)

                                            sentence  sentiment
0  awww that s a bummer you shoulda got david car...          0
1  is upset that he can t update his facebook by ...          0
2  i dived many times for the ball managed to sav...          0
3     my whole body feels itchy and like its on fire          0
4  no it s not behaving at all i m mad why am i h...          0
Index(['sentence', 'sentiment'], dtype='object')


In [7]:
import pandas as pd

train_path = "/kaggle/input/datasets/krishbaisoya/tweets-sentiment-analysis/train_data.csv"

df = pd.read_csv(train_path)

# Rename columns to match your BERT code
df = df.rename(columns={
    "sentence": "text",
    "sentiment": "target"
})

# Keep only required columns
df = df[["target", "text"]]

# Check labels
print(df.head())
print(df["target"].value_counts())

# Sample 100,000 rows (same as your Colab experiment)
df = df.sample(100_000, random_state=42)

print("Dataset shape:", df.shape)
print(df["target"].value_counts())

   target                                               text
0       0  awww that s a bummer you shoulda got david car...
1       0  is upset that he can t update his facebook by ...
2       0  i dived many times for the ball managed to sav...
3       0     my whole body feels itchy and like its on fire
4       0  no it s not behaving at all i m mad why am i h...
target
0    767059
1    756916
Name: count, dtype: int64
Dataset shape: (100000, 2)
target
0    50090
1    49910
Name: count, dtype: int64


### 1.3 Load Tokenizer and Model

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [9]:
print('model' in dir())
print('val_loader' in dir())
print('device' in dir())

False
False
True


In [10]:
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)
model.to(device)
print("Model loaded on", device)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda


### 1.4 Dataset Class, Train/Validation Split, and DataLoaders

In [11]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx])
        }


# This dataset only ships a single training file, so a held-out test set
# is carved out here from our own data (same reasoning as Sentiment140 --
# no separate test file means splitting is the only option).
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["target"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["target"].tolist()
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

train_dataset = SentimentDataset(train_texts, train_labels)
val_dataset   = SentimentDataset(val_texts,   val_labels)
test_dataset  = SentimentDataset(test_texts,  test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)
test_loader  = DataLoader(test_dataset,  batch_size=16)

print(f"Train: {len(train_texts):,} | Validation: {len(val_texts):,} | Test: {len(test_texts):,}")

Train: 80,000 | Validation: 10,000 | Test: 10,000


### 1.5 Training Loop

In [12]:
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    print(f"\nEpoch {epoch+1}/{epochs}")

    for i, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if i % 200 == 0:
            print(f"  Step {i} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")


Epoch 1/3
  Step 0 | Loss: 0.6898
  Step 200 | Loss: 0.3759
  Step 400 | Loss: 0.4626
  Step 600 | Loss: 0.4673
  Step 800 | Loss: 0.3554
  Step 1000 | Loss: 0.5993
  Step 1200 | Loss: 0.3167
  Step 1400 | Loss: 0.3168
  Step 1600 | Loss: 0.3671
  Step 1800 | Loss: 0.5221
  Step 2000 | Loss: 0.2763
  Step 2200 | Loss: 0.4161
  Step 2400 | Loss: 0.5183
  Step 2600 | Loss: 0.4258
  Step 2800 | Loss: 0.4100
  Step 3000 | Loss: 0.6298
  Step 3200 | Loss: 0.3493
  Step 3400 | Loss: 0.3171
  Step 3600 | Loss: 0.3421
  Step 3800 | Loss: 0.3839
  Step 4000 | Loss: 0.5497
  Step 4200 | Loss: 0.2832
  Step 4400 | Loss: 0.3708
  Step 4600 | Loss: 0.3507
  Step 4800 | Loss: 0.2325
Epoch 1 Average Loss: 0.4194

Epoch 2/3
  Step 0 | Loss: 0.1823
  Step 200 | Loss: 0.1345
  Step 400 | Loss: 0.1598
  Step 600 | Loss: 0.1839
  Step 800 | Loss: 0.3314
  Step 1000 | Loss: 0.3705
  Step 1200 | Loss: 0.1450
  Step 1400 | Loss: 0.1544
  Step 1600 | Loss: 0.1458
  Step 1800 | Loss: 0.3281
  Step 2000 | Loss

In [13]:
import torch
print("torch version:", torch.__version__)
print("torch CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("Supported archs:", torch.cuda.get_arch_list())

torch version: 2.10.0+cu128
torch CUDA version: 12.8
GPU: Tesla T4
Compute capability: (7, 5)
Supported archs: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [14]:
!nvidia-smi

Sat Aug  8 19:47:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             66W /   70W |    2435MiB /  15360MiB |     99%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1.6 Evaluation

In [15]:
def evaluate(loader, split_name):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            logits      = outputs.logits
            predictions = torch.argmax(logits, dim=1).cpu().numpy()

            preds.extend(predictions)
            true_labels.extend(batch["labels"].numpy())

    print(f"===== {split_name} =====")
    print(classification_report(true_labels, preds, target_names=["Negative", "Positive"]))
    print()


evaluate(val_loader, "Validation")
evaluate(test_loader, "Test (held out, never seen during training)")

===== Validation =====
              precision    recall  f1-score   support

    Negative       0.81      0.85      0.83      5009
    Positive       0.84      0.79      0.82      4991

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000


===== Test (held out, never seen during training) =====
              precision    recall  f1-score   support

    Negative       0.80      0.85      0.82      5009
    Positive       0.84      0.79      0.81      4991

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000




### 1.7 Inference on Test Sentences

The following sentences test the baseline model's ability to handle pragmatic language. The model is expected to struggle with sarcasm and irony — this motivates the design of Phase 2.


In [16]:
def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    label = "Positive" if probs[1] > probs[0] else "Negative"
    print(f"Text:      {text}")
    print(f"Sentiment: {label}  (Neg: {probs[0]:.3f}, Pos: {probs[1]:.3f})")
    print()


test_sentences = [
    "Oh great, my phone just died. Amazing.",
    "I just love being ignored.",
    "This movie was mid",
    "That party was lit!",
    "Wow, another rainy day. Perfect.",
    "I'm dead that was hilarious",
    "Best day ever... not"
]

for s in test_sentences:
    predict_sentiment(s)

Text:      Oh great, my phone just died. Amazing.
Sentiment: Negative  (Neg: 0.993, Pos: 0.007)

Text:      I just love being ignored.
Sentiment: Negative  (Neg: 0.750, Pos: 0.250)

Text:      This movie was mid
Sentiment: Positive  (Neg: 0.183, Pos: 0.817)

Text:      That party was lit!
Sentiment: Negative  (Neg: 0.970, Pos: 0.030)

Text:      Wow, another rainy day. Perfect.
Sentiment: Negative  (Neg: 0.712, Pos: 0.288)

Text:      I'm dead that was hilarious
Sentiment: Positive  (Neg: 0.292, Pos: 0.708)

Text:      Best day ever... not
Sentiment: Negative  (Neg: 0.937, Pos: 0.063)



Observation

The baseline model achieves 82% accuracy on the validation set (and 82% on the held-out test set), with a modest negative-leaning bias: it is more inclined to predict Negative than Positive when the signal is ambiguous (negative recall 0.85 vs positive recall 0.79 on both splits).

This bias is visible in the qualitative results above. The model correctly resolves cases anchored to explicit negative content or negation — "Oh great, my phone just died. Amazing." (Negative, 99.3%) and "Best day ever... not" (Negative, 93.7%). It also classifies "I just love being ignored." and "Wow, another rainy day. Perfect." as Negative (75.0% and 71.2% respectively), which is the intended sentiment for both — though given the model's general negative-leaning bias, this may reflect a default toward "negative when uncertain" as much as genuine sarcasm resolution, rather than a confident, high-margin call.

The clearer failures are on contemporary slang rather than sarcasm: "That party was lit!" (genuinely positive) is misclassified Negative at 97.0% confidence, and "This movie was mid" (lukewarm/negative) is misclassified Positive at 81.7% confidence. `bert-base-cased`'s pretraining corpus predates this vocabulary, which is a more likely explanation for these two errors than a failure to detect sarcasm specifically. This, together with the model's inability to reliably distinguish genuine sarcasm from surface-level polarity words, motivates Phase 2: a multi-task model that jointly detects sentiment and sarcasm using contrastive learning.


---
## Phase 2: Sarcasm-Aware Multi-Task Learning Model

A `MultiTaskBERT` architecture is built with a shared BERT encoder and two classification heads - one for sentiment, one for sarcasm detection. Training incorporates a contrastive loss component to help the model distinguish sarcastic from sincere language.

**Datasets used:**
- Sentiment140 (100,000 tweets)
- Sarcasm News Headlines (HuggingFace: `raquiba/Sarcasm_News_Headline`)
- SemEval-2018 Twitter Irony (`cardiffnlp/tweet_eval`, irony split)

> **Note on data pipeline corrections:**
> 1. Random seeds are fixed across Python, NumPy and PyTorch for reproducibility (Section 1.1).
> 2. Sentiment labels are not fabricated from the sarcasm label for the headline/irony datasets; those rows carry a `-100` placeholder and are excluded from the sentiment loss and sentiment evaluation via `ignore_index=-100`, keeping the two tasks statistically independent.
> 3. The train/validation/test split is performed **before** sarcastic-class upsampling, with upsampling applied to the training split only, so duplicated rows cannot leak into validation or test and inflate sarcasm accuracy.
> 4. A held-out **test set** is used throughout (rather than validation alone), with Section 2.7 reporting both validation and test metrics.
> 5. The contrastive loss operates on the shared CLS embedding rather than on the two-dimensional classifier logits, with a margin (0.3) suited to cosine similarity's [-1, 1] range.


### 2.1 Load Additional Datasets

In [17]:
!pip install datasets -q

from datasets import load_dataset

# NOTE ON A FIXED FLAW:
# Previously, sentiment_label for the sarcasm/irony datasets was derived
# directly from the sarcasm_label (sarcastic -> negative, not sarcastic -> positive).
# That's a strong, often-false assumption (sarcasm can carry positive surface
# sentiment, and vice versa), and it meant the "sentiment" task on these rows
# was really just a restatement of the sarcasm task -- confounding the two.
#
# Fix: these datasets only have reliable SARCASM labels. We mark their
# sentiment_label as -100 (a placeholder), and later use ignore_index=-100
# so the sentiment loss is only ever computed on rows with real sentiment
# labels (Sentiment140). This keeps the two tasks properly separated.

# -- Load Sarcasm News Headlines --------------------------------------------
sarcasm_dataset = load_dataset("raquiba/Sarcasm_News_Headline", split="train")
sarcasm_df = sarcasm_dataset.to_pandas()
sarcasm_df = sarcasm_df[["headline", "is_sarcastic"]]
sarcasm_df.columns = ["text", "sarcasm_label"]
sarcasm_df["sentiment_label"] = -100  # no reliable sentiment label available
print("Sarcasm News Headlines:", sarcasm_df.shape)

# -- Load SemEval-2018 Twitter Irony -----------------------------------------
irony_dataset = load_dataset("cardiffnlp/tweet_eval", "irony")
irony_df = irony_dataset["train"].to_pandas()
irony_df = irony_df[["text", "label"]]
irony_df.columns = ["text", "sarcasm_label"]
irony_df["sentiment_label"] = -100  # no reliable sentiment label available
print("SemEval Twitter Irony:", irony_df.shape)
print("Irony label distribution:", irony_df["sarcasm_label"].value_counts().to_dict())


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/28619 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26709 [00:00<?, ? examples/s]

... (5 outputs left out)

Generating train split:   0%|          | 0/2862 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/784 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/955 [00:00<?, ? examples/s]

SemEval Twitter Irony: (2862, 3)
Irony label distribution: {1: 1445, 0: 1417}


### 2.2 Merge and Rebalance Datasets

In [18]:
# -- Prepare Sentiment140 -----------------------------------------------------
sentiment140_df = df[["text", "target"]].copy()
sentiment140_df.columns = ["text", "sentiment_label"]
sentiment140_df["sarcasm_label"] = 0  # Sentiment140 tweets are treated as non-sarcastic

# -- Merge all three sources ---------------------------------------------------
combined_df = pd.concat([sentiment140_df, sarcasm_df, irony_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Combined shape:", combined_df.shape)
print(combined_df[["sarcasm_label"]].value_counts())
print("Rows with a real sentiment label:", (combined_df["sentiment_label"] != -100).sum())

# NOTE ON A FIXED FLAW:
# Upsampling used to happen HERE, before the train/val split. Because
# resample(replace=True) duplicates rows, duplicates could land in both the
# training set and the validation set -- i.e. the model could be evaluated
# on rows it had already been trained on, inflating validation accuracy.
# Rebalancing now happens in the next cell, AFTER the split, and only on
# the training portion.


Combined shape: (131481, 3)
sarcasm_label
0                116402
1                 15079
Name: count, dtype: int64
Rows with a real sentiment label: 100000


### 2.3 Multi-Task Dataset Class, Split, and DataLoaders

In [19]:
class MultiTaskDataset(Dataset):
    def __init__(self, texts, sentiment_labels, sarcasm_labels, tokenizer):
        self.texts            = texts
        self.sentiment_labels = sentiment_labels
        self.sarcasm_labels   = sarcasm_labels
        self.tokenizer        = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "sentiment":      torch.tensor(self.sentiment_labels[idx], dtype=torch.long),
            "sarcasm":        torch.tensor(self.sarcasm_labels[idx],   dtype=torch.long)
        }


# ---- Split BEFORE any upsampling, stratified on sarcasm_label -------------
train_df, temp_df = train_test_split(
    combined_df, test_size=0.2, random_state=42, stratify=combined_df["sarcasm_label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["sarcasm_label"]
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# ---- Rebalance the sarcastic class, TRAIN SPLIT ONLY -----------------------
train_sarcastic     = train_df[train_df["sarcasm_label"] == 1]
train_non_sarcastic = train_df[train_df["sarcasm_label"] == 0]

print(f"\nBefore rebalancing (train only) — sarcastic: {len(train_sarcastic):,}, "
      f"non-sarcastic: {len(train_non_sarcastic):,}")

train_sarcastic_upsampled = resample(
    train_sarcastic,
    replace=True,
    n_samples=len(train_non_sarcastic),
    random_state=42
)

train_balanced_df = pd.concat([train_non_sarcastic, train_sarcastic_upsampled])
train_balanced_df = train_balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"After rebalancing (train only) — sarcastic: "
      f"{len(train_balanced_df[train_balanced_df['sarcasm_label']==1]):,}, "
      f"non-sarcastic: {len(train_balanced_df[train_balanced_df['sarcasm_label']==0]):,}")

# val_df and test_df are left at their natural (non-upsampled) distribution,
# since we want validation/test performance to reflect real-world class balance.
print("Val sarcasm distribution:",  val_df["sarcasm_label"].value_counts().to_dict())
print("Test sarcasm distribution:", test_df["sarcasm_label"].value_counts().to_dict())

mt_train_dataset = MultiTaskDataset(
    train_balanced_df["text"].tolist(),
    train_balanced_df["sentiment_label"].tolist(),
    train_balanced_df["sarcasm_label"].tolist(),
    tokenizer
)
mt_val_dataset = MultiTaskDataset(
    val_df["text"].tolist(),
    val_df["sentiment_label"].tolist(),
    val_df["sarcasm_label"].tolist(),
    tokenizer
)
mt_test_dataset = MultiTaskDataset(
    test_df["text"].tolist(),
    test_df["sentiment_label"].tolist(),
    test_df["sarcasm_label"].tolist(),
    tokenizer
)

mt_train_loader = DataLoader(mt_train_dataset, batch_size=16, shuffle=True)
mt_val_loader   = DataLoader(mt_val_dataset,   batch_size=16)
mt_test_loader  = DataLoader(mt_test_dataset,  batch_size=16)

print(f"\nDataLoaders ready — train: {len(mt_train_dataset):,}, "
      f"val: {len(mt_val_dataset):,}, test: {len(mt_test_dataset):,}")

Train: 105,184 | Val: 13,148 | Test: 13,149

Before rebalancing (train only) — sarcastic: 12,063, non-sarcastic: 93,121
After rebalancing (train only) — sarcastic: 93,121, non-sarcastic: 93,121
Val sarcasm distribution: {0: 11640, 1: 1508}
Test sarcasm distribution: {0: 11641, 1: 1508}

DataLoaders ready — train: 186,242, val: 13,148, test: 13,149


### 2.4 MultiTaskBERT Architecture

In [20]:
class MultiTaskBERT(nn.Module):
    def __init__(self, num_sentiment_labels=2, num_sarcasm_labels=2):
        super(MultiTaskBERT, self).__init__()

        # Shared BERT encoder
        self.bert = BertModel.from_pretrained("bert-base-cased")

        # Task-specific classification heads
        self.sentiment_classifier = nn.Linear(self.bert.config.hidden_size, num_sentiment_labels)
        self.sarcasm_classifier   = nn.Linear(self.bert.config.hidden_size, num_sarcasm_labels)

        self.dropout = nn.Dropout(0.3)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # [CLS] token representation - summary of the whole sentence.
        # This shared embedding is what the contrastive loss now operates
        # on (previously the contrastive loss was applied to the 2-dim
        # classifier LOGITS instead, which aren't a meaningful embedding
        # space to pull together / push apart).
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        sentiment_logits = self.sentiment_classifier(cls_output)
        sarcasm_logits   = self.sarcasm_classifier(cls_output)

        return sentiment_logits, sarcasm_logits, cls_output


model = MultiTaskBERT()
model.to(device)
print("MultiTaskBERT ready on", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MultiTaskBERT ready on cuda


### 2.5 Contrastive Loss Function

In [21]:
def contrastive_loss(embeddings, sarcasm_labels, margin=0.3):
    """
    Batch-level pairwise contrastive loss over the SHARED BERT embedding.

    Fixes three problems in the original version:
    1. It compared each example's two classifier LOGIT vectors to each
       other -- not a shared embedding space, so "distance" between them
       had no real geometric meaning. It now compares the shared CLS
       embedding across pairs of examples in the batch.
    2. It treated "sentiment_label == sarcasm_label" as the similarity
       signal -- a coincidental match between two unrelated label spaces.
       It now uses "same sarcasm_label" as the similarity signal, which
       matches the model's actual goal: separating sarcastic from
       sincere language in embedding space.
    3. margin=1.0 with cosine similarity (range [-1, 1]) meant the
       dissimilar-pair term could basically never fire. margin is now
       0.3, so dissimilar pairs are actually penalised when their
       similarity creeps above that threshold.
    """
    embeddings = F.normalize(embeddings, dim=1)
    sim_matrix = torch.matmul(embeddings, embeddings.T)  # cosine similarity, since normalized

    same_sarcasm = (sarcasm_labels.unsqueeze(0) == sarcasm_labels.unsqueeze(1)).float()

    n = sarcasm_labels.size(0)
    off_diag = ~torch.eye(n, dtype=torch.bool, device=embeddings.device)

    pos_loss = same_sarcasm * (1 - sim_matrix) ** 2
    neg_loss = (1 - same_sarcasm) * torch.clamp(sim_matrix - margin, min=0) ** 2

    loss = (pos_loss + neg_loss)[off_diag]
    return loss.mean()

### 2.6 Training Loop

In [22]:
sentiment_loss_fn = CrossEntropyLoss(ignore_index=-100)
sarcasm_loss_fn   = CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    print(f"\nEpoch {epoch+1}/{epochs}")

    for i, batch in enumerate(mt_train_loader):
        input_ids        = batch["input_ids"].to(device)
        attention_mask   = batch["attention_mask"].to(device)
        sentiment_labels = batch["sentiment"].to(device)
        sarcasm_labels   = batch["sarcasm"].to(device)

        sentiment_logits, sarcasm_logits, cls_embeddings = model(input_ids, attention_mask)

        # Guard against batches with zero rows carrying a real sentiment label
        # (possible since ~50% of the batch is sarcasm/irony data with a -100
        # placeholder). CrossEntropyLoss(ignore_index=-100) computes 0/0 -> nan
        # in that case, which poisons the running loss total for the whole epoch.
        has_real_sentiment = (sentiment_labels != -100).any()
        if has_real_sentiment:
            sentiment_loss = sentiment_loss_fn(sentiment_logits, sentiment_labels)
        else:
            sentiment_loss = torch.tensor(0.0, device=device)

        sarcasm_loss = sarcasm_loss_fn(sarcasm_logits, sarcasm_labels)

        # Contrastive loss operates on the shared CLS embedding, using
        # sarcasm_label as the similarity signal.
        cl = contrastive_loss(cls_embeddings, sarcasm_labels, margin=0.3)

        total_loss_val = sentiment_loss + sarcasm_loss + 0.5 * cl

        optimizer.zero_grad()
        total_loss_val.backward()
        optimizer.step()

        total_loss += total_loss_val.item()

        if i % 200 == 0:
            print(f"  Step {i} | Sentiment: {sentiment_loss.item():.4f} | "
                  f"Sarcasm: {sarcasm_loss.item():.4f} | "
                  f"Contrastive: {cl.item():.4f} | "
                  f"Total: {total_loss_val.item():.4f}")

    avg = total_loss / len(mt_train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg:.4f}")


Epoch 1/3
  Step 0 | Sentiment: 0.5528 | Sarcasm: 0.7126 | Contrastive: 0.1293 | Total: 1.3300
  Step 200 | Sentiment: 0.7263 | Sarcasm: 0.5426 | Contrastive: 0.1867 | Total: 1.3623
  Step 400 | Sentiment: 0.5969 | Sarcasm: 0.3851 | Contrastive: 0.1310 | Total: 1.0475
  Step 600 | Sentiment: 0.4978 | Sarcasm: 0.0855 | Contrastive: 0.0895 | Total: 0.6281
  Step 800 | Sentiment: 0.1447 | Sarcasm: 0.0727 | Contrastive: 0.0877 | Total: 0.2612
  Step 1000 | Sentiment: 0.6731 | Sarcasm: 0.0613 | Contrastive: 0.0845 | Total: 0.7766
  Step 1200 | Sentiment: 0.2664 | Sarcasm: 0.0434 | Contrastive: 0.0785 | Total: 0.3489
  Step 1400 | Sentiment: 0.5178 | Sarcasm: 0.0252 | Contrastive: 0.0772 | Total: 0.5816
  Step 1600 | Sentiment: 0.2190 | Sarcasm: 0.2071 | Contrastive: 0.0976 | Total: 0.4749
  Step 1800 | Sentiment: 0.3319 | Sarcasm: 0.1850 | Contrastive: 0.1022 | Total: 0.5680
  Step 2000 | Sentiment: 0.3359 | Sarcasm: 0.0188 | Contrastive: 0.0830 | Total: 0.3963
  Step 2200 | Sentiment: 0.3

### 2.7 Evaluation

In [23]:
def evaluate(loader, split_name, eval_model):
    eval_model.eval()
    all_sent_preds, all_sent_true = [], []
    all_sarc_preds, all_sarc_true = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)

            sent_logits, sarc_logits, _ = eval_model(input_ids, attn_mask)

            all_sent_preds.extend(torch.argmax(sent_logits, dim=1).cpu().numpy())
            all_sent_true.extend(batch["sentiment"].numpy())

            all_sarc_preds.extend(torch.argmax(sarc_logits, dim=1).cpu().numpy())
            all_sarc_true.extend(batch["sarcasm"].numpy())

    all_sent_preds = np.array(all_sent_preds)
    all_sent_true  = np.array(all_sent_true)
    sent_mask = all_sent_true != -100

    print(f"===== {split_name} =====")
    print("--- Sentiment Head "
          f"(scored on {sent_mask.sum():,} of {len(all_sent_true):,} rows "
          "with a real sentiment label) ---")
    print(classification_report(
        all_sent_true[sent_mask], all_sent_preds[sent_mask],
        target_names=["Negative", "Positive"]
    ))

    print("--- Sarcasm Head ---")
    print(classification_report(
        all_sarc_true, all_sarc_preds,
        target_names=["Not Sarcastic", "Sarcastic"]
    ))
    print()

In [24]:
# ---- Ablation: multi-task model WITHOUT the contrastive loss term ----
# Identical to the full model in every other way (same data, architecture,
# hyperparameters). Dropping only the 0.5 * cl term isolates what the
# contrastive loss is actually contributing.

model_ablation = MultiTaskBERT()
model_ablation.to(device)

sentiment_loss_fn = CrossEntropyLoss(ignore_index=-100)
sarcasm_loss_fn   = CrossEntropyLoss()
optimizer_ablation = AdamW(model_ablation.parameters(), lr=2e-5)
epochs = 3

for epoch in range(epochs):
    model_ablation.train()
    total_loss = 0
    print(f"\n[Ablation] Epoch {epoch+1}/{epochs}")

    for i, batch in enumerate(mt_train_loader):
        input_ids        = batch["input_ids"].to(device)
        attention_mask   = batch["attention_mask"].to(device)
        sentiment_labels = batch["sentiment"].to(device)
        sarcasm_labels   = batch["sarcasm"].to(device)

        sentiment_logits, sarcasm_logits, _ = model_ablation(input_ids, attention_mask)

        # Guard against batches with zero rows carrying a real sentiment label
        # (possible since ~50% of the batch is sarcasm/irony data with a -100
        # placeholder). CrossEntropyLoss(ignore_index=-100) computes 0/0 -> nan
        # in that case, which poisons the running loss total for the whole epoch.
        has_real_sentiment = (sentiment_labels != -100).any()
        if has_real_sentiment:
            sentiment_loss = sentiment_loss_fn(sentiment_logits, sentiment_labels)
        else:
            sentiment_loss = torch.tensor(0.0, device=device)

        sarcasm_loss = sarcasm_loss_fn(sarcasm_logits, sarcasm_labels)

        # No contrastive term here -- this is the entire point of the ablation
        total_loss_val = sentiment_loss + sarcasm_loss

        optimizer_ablation.zero_grad()
        total_loss_val.backward()
        optimizer_ablation.step()

        total_loss += total_loss_val.item()

        if i % 200 == 0:
            print(f"  Step {i} | Sentiment: {sentiment_loss.item():.4f} | "
                  f"Sarcasm: {sarcasm_loss.item():.4f} | "
                  f"Total: {total_loss_val.item():.4f}")

    avg = total_loss / len(mt_train_loader)
    print(f"[Ablation] Epoch {epoch+1} Average Loss: {avg:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Ablation] Epoch 1/3
  Step 0 | Sentiment: 0.7368 | Sarcasm: 0.7221 | Total: 1.4589
  Step 200 | Sentiment: 0.3994 | Sarcasm: 0.2135 | Total: 0.6130
  Step 400 | Sentiment: 0.8307 | Sarcasm: 0.2731 | Total: 1.1038
  Step 600 | Sentiment: 0.6323 | Sarcasm: 0.3352 | Total: 0.9674
  Step 800 | Sentiment: 0.0887 | Sarcasm: 0.0706 | Total: 0.1594
  Step 1000 | Sentiment: 0.5618 | Sarcasm: 0.2243 | Total: 0.7861
  Step 1200 | Sentiment: 0.5004 | Sarcasm: 0.2044 | Total: 0.7048
  Step 1400 | Sentiment: 0.1748 | Sarcasm: 0.0430 | Total: 0.2178
  Step 1600 | Sentiment: 0.6365 | Sarcasm: 0.3447 | Total: 0.9812
  Step 1800 | Sentiment: 0.7062 | Sarcasm: 0.0992 | Total: 0.8054
  Step 2000 | Sentiment: 0.8324 | Sarcasm: 0.0641 | Total: 0.8965
  Step 2200 | Sentiment: 0.4794 | Sarcasm: 0.0163 | Total: 0.4957
  Step 2400 | Sentiment: 0.7053 | Sarcasm: 0.0729 | Total: 0.7782
  Step 2600 | Sentiment: 0.1944 | Sarcasm: 0.1046 | Total: 0.2991
  Step 2800 | Sentiment: 0.6629 | Sarcasm: 0.3424 | Total: 1.

In [25]:
evaluate(mt_test_loader, "Full model (with contrastive loss) - Test", model)
evaluate(mt_test_loader, "Ablation (no contrastive loss) - Test", model_ablation)

===== Full model (with contrastive loss) - Test =====
--- Sentiment Head (scored on 10,007 of 13,149 rows with a real sentiment label) ---
              precision    recall  f1-score   support

    Negative       0.81      0.85      0.83      5019
    Positive       0.84      0.80      0.82      4988

    accuracy                           0.83     10007
   macro avg       0.83      0.83      0.83     10007
weighted avg       0.83      0.83      0.83     10007

--- Sarcasm Head ---
               precision    recall  f1-score   support

Not Sarcastic       0.99      0.98      0.98     11641
    Sarcastic       0.84      0.92      0.88      1508

     accuracy                           0.97     13149
    macro avg       0.92      0.95      0.93     13149
 weighted avg       0.97      0.97      0.97     13149


===== Ablation (no contrastive loss) - Test =====
--- Sentiment Head (scored on 10,007 of 13,149 rows with a real sentiment label) ---
              precision    recall  f1-score 

---
## Summary

### Phase 1 - Baseline Sentiment Model
- Fine-tuned `bert-base-cased` on 100,000 tweets from the Sentiment140 dataset
- Binary classification: Positive / Negative
- Training loss: 0.42 → 0.31 → 0.20 over 3 epochs
- Validation accuracy: **82%** | Test accuracy: **82%**
- Limitation: negative-leaning bias, and unreliable on contemporary slang not seen in pretraining (e.g. misclassifies *"That party was lit!"* as Negative)

### Phase 2 - Multi-Task Sarcasm-Aware Model
- Shared BERT encoder with two task-specific classification heads (sentiment + sarcasm)
- Combined loss: `sentiment_loss + sarcasm_loss + 0.5 × contrastive_loss`
- Training data: Sentiment140 + Sarcasm News Headlines + SemEval-2018 Twitter Irony (131,481 rows combined; sarcastic class upsampled to balance the training split only)
- Full model (with contrastive loss) test accuracy: **83%** sentiment / **97%** sarcasm — vs. **81%** sentiment for an otherwise-identical ablation without the contrastive term, and **82%** for the Phase 1 baseline
- The contrastive loss term also raises sarcasm-class recall (0.92 vs 0.89 for the ablation), at a modest precision cost, with no change to overall sarcasm F1 (0.88 for both)
- Known limitation: single-seed run, no confidence interval on the 2-point sentiment-accuracy gap; cross-domain sarcasm detection (news-style vs. conversational sarcasm) remains a challenge
